# Enhanced Model for Sign Language Detection with Deep CNN
### Official code: DCNN v2 for ASL hand sign recognition

Paper: A. Rehman, M. Zaman, T. Kehkashan, F. Akbar, M. Hamza and R. A. Riaz, "Enhanced Model for Sign Language Detection with Deep CNN: Achieving Perfect Accuracy in Hand Gesture Recognition", 2024 5th International Conference on Innovative Computing (ICIC), IEEE. DOI: [10.1109/ICIC63915.2024.11116573](https://doi.org/10.1109/ICIC63915.2024.11116573)

> **Note.** The original training notebook used for the paper is no longer available. This notebook was rewritten from the method described in the paper (Section III, Table I and Fig. 1) and has not yet been re-run, so results may differ from those reported. Settings that the paper does not specify are marked as assumptions below.

Pipeline:

1. **Data**: American Hand Sign Detection dataset (Kaggle), 26 classes (A to Z), 200x200 RGB images.
2. **Preprocessing**: resize to 64x64, normalize pixels to [0, 1], data augmentation (brightness, contrast, sharpness, zoom).
3. **Model**: DCNN v2 as drawn in Fig. 1 of the paper.
4. **Training**: Stratified K-Fold cross-validation with the settings in Table I.
5. **Evaluation**: accuracy, precision, recall, F1-score, training curves, confusion matrix, ROC curve, example predictions.

| Setting | Value | Source |
|---|---|---|
| Input size | 64x64x3 | Section III-B, Fig. 1 |
| Conv2D layers | 64 (7x7), 128, 256, 512, 1024 (3x3) filters, ReLU | Section III-C, Fig. 1 |
| Max pooling | 3x3, after the first convolution | Fig. 1 |
| Batch normalization | after every convolution | Section III-C, Fig. 1 |
| Head | Global average pooling, Dense + ReLU, Dropout, Dense + softmax (26) | Section III-C, Fig. 1 |
| Cross-validation | Stratified K-Fold, 8 folds | Table I |
| Epochs / batch size | 15 / 32 | Table I |
| Optimizer / learning rate | Adam / 0.001 | Table I |
| Seed | 42 | Table I |
| Padding, strides | `same` padding, stride 1; pooling stride equal to its size (Keras default) | **Assumption** |
| Dense units / dropout rate | 256 / 0.5 | **Assumption** |
| Augmentation strengths | see Section 3 | **Assumption** |
| Loss | categorical cross-entropy | **Assumption** (standard for softmax classification) |

## 0. Setup
Install the dependencies once from the repository root with `pip install -r requirements.txt`. Figures are written to `results/`, relative to this notebook's working directory.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import keras
from keras import layers
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support, classification_report,
                             confusion_matrix, roc_auc_score, roc_curve)

# Training settings from Table I of the paper
SEED = 42
N_FOLDS = 8
EPOCHS = 15
BATCH_SIZE = 32
LEARNING_RATE = 0.001
IMG_SIZE = 64            # 200x200 source images are resized to 64x64 (Section III-B, Fig. 1)

# Not specified in the paper: assumptions of this implementation
DENSE_UNITS = 256
DROPOUT_RATE = 0.5

DATA_DIR = Path("../data/asl")   # one sub-folder per letter; see the Dataset section of the README
RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)

keras.utils.set_random_seed(SEED)
print("Keras", keras.__version__, "| backend:", keras.backend.backend())

## 1. Dataset
American Hand Sign Detection dataset from Kaggle: <https://www.kaggle.com/datasets/konami9889/american-hand-sign-detection>. Download it and extract it into `data/asl/` at the repository root.

Each image is labelled with the name of the folder that contains it. If the download has separate train and validation folders, all images are pooled, because the paper evaluates with Stratified K-Fold cross-validation (Table I).

In [ ]:
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp"}

def load_dataset(root, img_size):
    paths = sorted(p for p in Path(root).rglob("*") if p.suffix.lower() in IMG_EXTS)
    if not paths:
        raise FileNotFoundError(f"No images found under {Path(root).resolve()}. See the Dataset section of the README.")
    labels = np.array([p.parent.name for p in paths])
    # Resize from 200x200 to img_size x img_size while loading (Section III-B, step 1)
    images = np.stack([np.asarray(keras.utils.load_img(p, color_mode="rgb", target_size=(img_size, img_size),
                                                       interpolation="bilinear")) for p in paths])
    return images.astype("uint8"), labels

X, label_names = load_dataset(DATA_DIR, IMG_SIZE)
class_names = sorted(set(label_names))
y = np.array([class_names.index(name) for name in label_names])
NUM_CLASSES = len(class_names)

print("Images:", X.shape, "| classes:", NUM_CLASSES)
print("Classes:", ", ".join(class_names))

In [ ]:
counts = pd.Series(label_names).value_counts().reindex(class_names)
counts.plot.bar(figsize=(12, 3.5), color="#4C72B0")
plt.title("Images per class"); plt.ylabel("Count"); plt.tight_layout(); plt.show()

rng = np.random.default_rng(SEED)
idx = rng.choice(len(X), size=min(16, len(X)), replace=False)
fig, axes = plt.subplots(2, 8, figsize=(14, 4))
for ax, i in zip(axes.ravel(), idx):
    ax.imshow(X[i]); ax.set_title(class_names[y[i]]); ax.axis("off")
plt.suptitle("Sample images (64x64)"); plt.tight_layout(); plt.show()

## 2. Preprocessing and DCNN v2 architecture
Normalization (Section III-B, step 2) and augmentation (step 3) are layers at the start of the model. Augmentation is only active during training. The paper names brightness, sharpness and zoom (and contrast in Section I). The strengths below are assumptions.

The layer order follows Fig. 1 of the paper: Conv2D(64, 7x7) + ReLU, MaxPooling(3x3), BatchNorm, then Conv2D(3x3) + ReLU and BatchNorm blocks with 128, 256, 512 and 1024 filters, then global average pooling, Dense + ReLU with dropout, and a softmax output.

In [ ]:
def make_augmentation():
    return keras.Sequential([
        layers.RandomBrightness(0.2, value_range=(0.0, 1.0)),
        layers.RandomContrast(0.2),
        layers.RandomSharpness((0.3, 0.7), value_range=(0.0, 1.0)),   # 0.5 = unchanged
        layers.RandomZoom(0.1),
    ], name="augmentation")


def build_dcnn_v2(num_classes):
    inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x = layers.Rescaling(1.0 / 255)(inputs)          # pixel values in [0, 1]
    x = make_augmentation()(x)

    x = layers.Conv2D(64, 7, padding="same", activation="relu")(x)
    x = layers.MaxPooling2D(3)(x)
    x = layers.BatchNormalization()(x)
    for filters in (128, 256, 512, 1024):
        x = layers.Conv2D(filters, 3, padding="same", activation="relu")(x)
        x = layers.BatchNormalization()(x)

    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(DENSE_UNITS, activation="relu")(x)
    x = layers.Dropout(DROPOUT_RATE)(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)

    model = keras.Model(inputs, outputs, name="DCNN_v2")
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
                  loss="categorical_crossentropy",
                  metrics=["accuracy", keras.metrics.Precision(name="precision"), keras.metrics.Recall(name="recall")])
    return model


build_dcnn_v2(NUM_CLASSES).summary()

## 3. Training with Stratified K-Fold cross-validation (Table I)
8 folds, 15 epochs, batch size 32, Adam with learning rate 0.001, seed 42. A new model is trained for each fold, and the held-out fold is used for validation. Predictions on each held-out fold are collected so that every image is evaluated exactly once.

In [ ]:
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
Y = keras.utils.to_categorical(y, NUM_CLASSES)
oof_proba = np.zeros((len(X), NUM_CLASSES), dtype="float32")
histories, fold_rows = [], []

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), start=1):
    print(f"\n===== Fold {fold}/{N_FOLDS}: train={len(train_idx)}  validation={len(val_idx)} =====")
    keras.backend.clear_session()
    keras.utils.set_random_seed(SEED)
    model = build_dcnn_v2(NUM_CLASSES)
    history = model.fit(X[train_idx], Y[train_idx], validation_data=(X[val_idx], Y[val_idx]),
                        epochs=EPOCHS, batch_size=BATCH_SIZE, verbose=2)
    histories.append(history.history)

    proba = model.predict(X[val_idx], batch_size=BATCH_SIZE, verbose=0)
    oof_proba[val_idx] = proba
    pred = proba.argmax(axis=1)
    precision, recall, f1, _ = precision_recall_fscore_support(y[val_idx], pred, average="macro", zero_division=0)
    fold_rows.append({"Fold": fold, "Accuracy": accuracy_score(y[val_idx], pred),
                      "Precision": precision, "Recall": recall, "F1-score": f1})

fold_df = pd.DataFrame(fold_rows).set_index("Fold")
display(fold_df.style.format("{:.4f}"))
print("Mean over folds:")
for metric in fold_df.columns:
    print(f"  {metric:<10}: {fold_df[metric].mean():.4f} ± {fold_df[metric].std():.4f}")

## 4. Training and validation curves (Figs. 2 and 3)
Mean over folds, with each fold shown as a thin line.

In [ ]:
def plot_metric(key, title, fname):
    plt.figure(figsize=(8, 5))
    epochs = np.arange(1, EPOCHS + 1)
    for prefix, label, color in [("", "Training", "C0"), ("val_", "Validation", "C1")]:
        runs = np.array([h[prefix + key] for h in histories])
        for run in runs:
            plt.plot(epochs, run, color=color, alpha=0.15, lw=1)
        plt.plot(epochs, runs.mean(axis=0), color=color, lw=2, label=f"{label} {key}")
    plt.xlabel("Epoch"); plt.ylabel(key.capitalize()); plt.title(title)
    plt.legend(); plt.tight_layout(); plt.savefig(RESULTS_DIR / fname, dpi=300); plt.show()

plot_metric("loss", "Training and Validation Loss", "fig_loss.png")
plot_metric("accuracy", "Training and Validation Accuracy", "fig_accuracy.png")
plot_metric("precision", "Training and Validation Precision", "fig_precision.png")
plot_metric("recall", "Training and Validation Recall", "fig_recall.png")

## 5. Evaluation on the held-out folds
Per-class precision, recall and F1-score, and the confusion matrix, computed from the held-out predictions of all folds.

In [ ]:
oof_pred = oof_proba.argmax(axis=1)
print(classification_report(y, oof_pred, target_names=class_names, digits=4, zero_division=0))

cm = confusion_matrix(y, oof_pred)
plt.figure(figsize=(11, 9))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=class_names, yticklabels=class_names, cbar=False)
plt.xlabel("Predicted label"); plt.ylabel("True label"); plt.title("Confusion Matrix for Hand Sign Classification")
plt.tight_layout(); plt.savefig(RESULTS_DIR / "fig_confusion_matrix.png", dpi=300); plt.show()

## 6. ROC curve
One-vs-rest ROC curves for every class and the micro-average, from the held-out predictions.

In [ ]:
macro_auc = roc_auc_score(Y, oof_proba, multi_class="ovr", average="macro")
fpr_micro, tpr_micro, _ = roc_curve(Y.ravel(), oof_proba.ravel())

plt.figure(figsize=(7, 6))
for k in range(NUM_CLASSES):
    fpr, tpr, _ = roc_curve(Y[:, k], oof_proba[:, k])
    plt.plot(fpr, tpr, color="grey", alpha=0.3, lw=1)
plt.plot(fpr_micro, tpr_micro, color="C0", lw=2.5, label=f"Micro-average (macro AUC = {macro_auc:.4f})")
plt.plot([0, 1], [0, 1], "k--", lw=1)
plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate")
plt.title("Receiver Operating Characteristic (ROC) Curve")
plt.legend(loc="lower right"); plt.tight_layout(); plt.savefig(RESULTS_DIR / "fig_roc.png", dpi=300); plt.show()

## 7. Example predictions

In [ ]:
idx = np.random.default_rng(SEED).choice(len(X), size=min(16, len(X)), replace=False)
fig, axes = plt.subplots(2, 8, figsize=(14, 4.5))
for ax, i in zip(axes.ravel(), idx):
    ok = oof_pred[i] == y[i]
    ax.imshow(X[i]); ax.axis("off")
    ax.set_title(f"True: {class_names[y[i]]}\nPred: {class_names[oof_pred[i]]}", color="green" if ok else "red", fontsize=9)
plt.suptitle("Predictions on held-out images"); plt.tight_layout()
plt.savefig(RESULTS_DIR / "fig_predictions.png", dpi=300); plt.show()

## 8. Comparison with the paper (Table II)
The other rows are the values reported in the paper (in %). The first row is this run's cross-validation mean.

In [ ]:
comparison = pd.DataFrame([
    ["DCNN v2 (this run)", *(100 * fold_df.mean()).round(1).tolist()],
    ["DCNN v2 (reported in the paper)", 100, 100, 100, 100],
    ["CNN+RTS (baseline, reported)", 98, 96, 97.1, 96.4],
    ["CNN (reported)", 90, 98, 98, 98],
    ["CNN (reported)", 96.3, 96.8, 97.3, 96.2],
    ["CNN (reported)", 92.9, 93.2, 93.6, 93.1],
], columns=["Model", "Accuracy", "Precision", "Recall", "F1-score"])
comparison